In [1]:
#Import libraries
import time
import os
import requests
import io
from sys import platform

#import helper libraries
import pandas as pd
from urllib.parse import urlparse
from PIL import Image

#import selenium drivers
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException,WebDriverException, SessionNotCreatedException

def webdriver_executable():
    if platform == "linux" or platform == "linux2" or platform == "darwin":
        return 'chromedriver'
    return 'chromedriver.exe'

#Parameters
number_of_images = 10               # Desired number of images
headless = False                    # True = No Chrome GUI
min_resolution = (0, 0)             # Minimum desired image resolution
max_resolution = (9999, 9999)       # Maximum desired image resolution
max_missed = 10                     # Max number of failed images before exit
number_of_workers = 8               # Number of "workers" used
keep_filenames = False              # Keep original URL image filenames

#Define file path
webdriver_path = os.path.normpath(os.path.join(os.getcwd(), 'webdriver', webdriver_executable()))
image_path = os.path.normpath(os.path.join(os.getcwd(), 'photos'))
cwd = os.getcwd()
WORKING_DIRECTORY = os.path.dirname(os.path.dirname(os.path.dirname(cwd))) #Parent directory
excercise_path= os.path.join(WORKING_DIRECTORY,"0_Module\\M3_Machine_Learning\\Project\\data\\01-raw\\Excercises.csv")
print(excercise_path)


h:\Documents\My Training\CAS\CAS_DLS\0_Module\M3_Machine_Learning\Project\data\01-raw\Excercises.csv


In [2]:

#Generate Search Keys
df = pd.read_csv(excercise_path,encoding="utf-8",header=0)
l1_search_keys = df[(df["ID"]>0) & (df["ID"]<119)]["Exercise"].to_list()
l2_search_keys ="african,asian,european"
l3_search_keys ="female,male"
search_keys = []

prefix = "gym "
suffix = " -lingerie -nude"

for i in l1_search_keys:
    for j in l2_search_keys.split(","):
        for k in l3_search_keys.split(","):
            search_keys.append(prefix + i +" " + j.strip() + " " + k.strip() +suffix)




In [3]:
class GoogleImageScraper():
    def __init__(self, webdriver_path, image_path, search_key="cat", number_of_images=1, headless=True, min_resolution=(0, 0), max_resolution=(1920, 1080), max_missed=10):
        #check parameter types
        image_path = os.path.join(image_path, search_key)
        for element in l3_search_keys.split(","):
            image_path = image_path.replace(element,"")
        for element in l2_search_keys.split(","):
            image_path = image_path.replace(element,"")
        
        image_path= image_path.replace(suffix,"").replace(prefix,"").strip()
        
        if (type(number_of_images)!=int):
            print("[Error] Number of images must be integer value.")
            return
        if not os.path.exists(image_path):
            print("Image path not found. Creating a new folder.")
            os.makedirs(image_path)
            

        options = Options()
        if(headless):
            options.add_argument('--headless')
        driver = webdriver.Chrome()
        driver.set_window_size(1400,1050)
        driver.get("https://www.google.com")
 
        WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.ID, "W0wltc"))).click()     

        self.driver = driver
        self.search_key = search_key
        self.number_of_images = number_of_images
        self.webdriver_path = webdriver_path
        self.image_path = image_path
        self.url = "https://www.google.com/search?q=%s&num=10&sca_esv=414ecf1566e10a89&udm=2"%(search_key)
        self.headless=headless
        self.min_resolution = min_resolution
        self.max_resolution = max_resolution
        self.max_missed = max_missed

    def find_image_urls(self):
        self.driver.get(self.url)
        image_urls=[]
        count = 0
        missed_count = 0
        time.sleep(3)
        
        imgurl = self.driver.find_elements(By.CLASS_NAME, "F0uyec")
        for element in imgurl:
            try:
                element.click()
                missed_count = 0
                time.sleep(1)
                images = self.driver.find_element(By.XPATH, "(//img[@jsname='kn3ccd'])")
                image_urls.append(images.get_attribute("src"))
                count +=1
            except Exception:
                try:
                    images = self.driver.find_element(By.XPATH, "(//img[@jsname='JuXqh'])")
                    image_urls.append(images.get_attribute("src"))
                    count +=1
                except Exception:
                    print("not found")
                    time.sleep(30)
                    missed_count += 1
            if self.number_of_images < count or missed_count > self.max_missed:
                break
        print(image_urls)

        self.driver.quit()
        return image_urls

    def save_images(self,image_urls, keep_filenames):
        for indx,image_url in enumerate(image_urls):
            try:
                print("Image url:%s"%(image_url))
                search_string = ''.join(e for e in self.search_key.replace(suffix,"").replace(prefix,"").strip() if e.isalnum())
                image = requests.get(image_url,timeout=5)
                if image.status_code == 200:
                    with Image.open(io.BytesIO(image.content)) as image_from_web:
                        try:
                            if (keep_filenames):
                                #extact filename without extension from URL
                                o = urlparse(image_url)
                                image_url = o.scheme + "://" + o.netloc + o.path
                                name = os.path.splitext(os.path.basename(image_url))[0]
                                #join filename and extension
                                filename = "%s.%s"%(name,image_from_web.format.lower())
                            else:
                                filename = "%s%s.%s"%(search_string,str(indx),image_from_web.format.lower())
                                for element in l3_search_keys.split(","):
                                    filename = filename.replace(element,str(l3_search_keys.split(",").index(element)))
                                for element in l2_search_keys.split(","):
                                    filename = filename.replace(element,str(l2_search_keys.split(",").index(element)))

                            image_path = os.path.join(self.image_path, filename)
                            print(f"{self.search_key} \t {indx} \t Image saved at: {image_path}")
                            image_from_web.save(image_path)
                        except OSError:
                            rgb_im = image_from_web.convert('RGB')
                            rgb_im.save(image_path)
                        image_resolution = image_from_web.size
                        if image_resolution != None:
                            if image_resolution[0]<self.min_resolution[0] or image_resolution[1]<self.min_resolution[1] or image_resolution[0]>self.max_resolution[0] or image_resolution[1]>self.max_resolution[1]:
                                image_from_web.close()
                                os.remove(image_path)

                        image_from_web.close()
            except Exception as e:
                print("[ERROR] Download failed: ",e)
                pass

In [12]:
def worker_thread(search_key):
    image_scraper = GoogleImageScraper(
        webdriver_path, 
        image_path, 
        search_key, 
        number_of_images, 
        headless, 
        min_resolution, 
        max_resolution, 
        max_missed)
    image_urls = image_scraper.find_image_urls()
    image_scraper.save_images(image_urls, keep_filenames)


#Run each search_key in a separate thread
#Automatically waits for all threads to finish
#Removes duplicate strings from search_keys
with concurrent.futures.ThreadPoolExecutor(max_workers=number_of_workers) as executor:
    executor.map(worker_thread, search_keys)

Image path not found. Creating a new folder.
Image path not found. Creating a new folder.
Image path not found. Creating a new folder.
['https://media.istockphoto.com/id/1310261421/photo/asian-man-squatting-in-a-training-gym.jpg?s=612x612&w=0&k=20&c=dgj-NO0TGplUd6psax3xQBnaDZ4_2IEuv7jFruBiUTU=', 'https://www.shutterstock.com/image-photo/young-asian-man-engaged-strength-260nw-2656851715.jpg', 'https://thumbs.dreamstime.com/b/asian-muscular-man-guy-performing-squat-exercise-legs-heavy-weights-barbell-gym-interior-male-doing-lifting-barbell-341055076.jpg', 'https://flex-web-media-prod.storage.googleapis.com/2024/10/asian-squat-yoga.jpg', 'https://thumbs.dreamstime.com/b/sport-man-doing-squat-posture-yoga-mat-fitness-gym-condominium-urban-people-lifestyles-workout-concept-149836387.jpg', 'https://www.shutterstock.com/image-photo/asian-muscular-man-guy-performing-260nw-2531337007.jpg', 'https://media.istockphoto.com/id/1398667091/photo/asian-muscular-strong-fit-man-biceps-squat-exercise-by-

In [ ]:
#for debugging only:
for search_key in search_keys:
    image_scraper = GoogleImageScraper(webdriver_path,image_path,search_key,number_of_images,headless,min_resolution,max_resolution)
    image_urls = image_scraper.find_image_urls()
    image_scraper.save_images(image_urls,keep_filenames=False)